In [ ]:
"""
compare_3d_roundnesses_supplement.ipynb
Author: Henry Yuan
Description: Code for plots in Figure S1 of paper supplemental material.
Plots values of roundness metrics from Bullard and Garboczi 2013 and Pieta et al. 2025
"""

# imports
import numpy as np
import sys
import matplotlib.pyplot as plt
sys.path.append('../../src')
import pandas as pd
from format_plots import format_plots

CR, CS, CE, CL, CRD, CED, CWAD, CSTD = format_plots() # format plots and collect colors

In [ ]:
# load roundness data for synthetic grains
data_file = '../../data/synthetic/random_synthetics/synthetic_stats.csv'
synthetic_stats_df = pd.read_csv(data_file)

# unpack into original roundness stage (round_stages), vertex count (nvs), and desired roundness metrics
metric_names = ['bg2013','fast_rs']
round_stages = synthetic_stats_df['roundness']
nvs = synthetic_stats_df['vertex_count']
bg = synthetic_stats_df['bg2013']
fast_rs = synthetic_stats_df['fast_rs']

In [ ]:
# function for more plot formatting, specific to this figure
def set_plot_layout():
    fig, axs = plt.subplots(3,1,figsize=(3.46,6),sharex=True)
    axs = axs.flatten()
    axs[2].set_xlabel('Mesh vertex count')

    for ax in axs:
        ax.set_xlim(500,10500)   # axis starts at 1000 (lowest x)
        ax.set_xticks(np.arange(500,10501,500))  # explicit ticks, first one is 1000
        ticks = ax.get_xticks()
        ax.set_xticklabels([f'{int(t)}' if (i % 4 == 1) else '' for i, t in enumerate(ticks)])
        ax.axvline(1000,ls='--',lw=1,color='black',alpha=0.5)
        ax.axvline(5000,ls='--',lw=1,color='black')

    normal_length = plt.rcParams["xtick.major.size"]  # default 3.5
    half_length = normal_length / 2

    # Draw the figure once so tick objects are fully created
    fig.canvas.draw()

    for ax in axs:
        for tick in ax.xaxis.get_major_ticks():
            value = tick.get_loc()
            if np.isclose(value % 1000, 0) or np.isclose(value % 1000, 1000):
                tick.tick1line.set_markersize(normal_length)  # bottom tick
                tick.tick2line.set_markersize(normal_length)  # top tick (if visible)
            else:
                tick.tick1line.set_markersize(half_length)
                tick.tick2line.set_markersize(half_length)

    return fig, axs

In [ ]:
# PLOT ROUNDNESS VS RESOLUTION AND RANGE EVOLUTION

# set plot layout
fig, axs = set_plot_layout()
alpha = 0.5

cs = [CE,CS,CR]

metric_labels = [r'$R_{BG}$',r'$R_P$']
rs = [0,0.5,1]
r_labels = ['euhedral','part round','round']

# label y-axes
for i,m in enumerate(metric_labels):
    axs[i].set_ylabel(m)

# loop through all roundness stages
for i,r in enumerate(rs):
    # subscript _i means "specific for this roundness stage"

    # get bool index list of all grains with roundness = r
    bool_list_i = (round_stages == r)
    # index into arrays to get subset of entire arrays for this specific r value
    nvs_i = nvs[bool_list_i]
    bg_i = bg[bool_list_i]
    fast_rs_i = fast_rs[bool_list_i]

    # scatter plot of all synthetics for wadell and std_curv
    axs[0].scatter(nvs_i, bg_i,color=cs[i],label=r_labels[i],s=1,alpha=alpha)
    axs[1].scatter(nvs_i, fast_rs_i,color=cs[i],label=r_labels[i],s=1,alpha=alpha)

axs[0].legend()

# reset ylims; manually chosen values
ylims = axs[0].get_ylim()
axs[0].set_ylim((ylims[0],1.05))
ylims = axs[1].get_ylim()
axs[1].set_ylim((ylims[0],1.05))

plt.tight_layout()
plt.savefig('figureS1.pdf')